# Aula 3 · Lab 2 — Produtores: chaves, partições e garantias

**Objetivos:**
- ver que mensagens com a **mesma chave vão sempre para a mesma partição** (e por isso ficam ordenadas);
- entender `acks`, idempotência e o callback de entrega;
- medir o efeito de `linger.ms` e `batch.size` no throughput (o trade-off latência × throughput da Aula 2).

In [ ]:
import json, time, uuid
from collections import Counter, defaultdict
from confluent_kafka import Producer
from comum.config import BOOTSTRAP

TOPICO = "aula3-pedidos"

## 1. Primeira mensagem e o callback de entrega
`produce()` é **assíncrono**: só coloca a mensagem num buffer. O resultado (partição e offset,
ou erro) chega no callback quando chamamos `poll()` ou `flush()`.

In [ ]:
produtor = Producer({"bootstrap.servers": BOOTSTRAP, "acks": "all", "enable.idempotence": True})

def ao_entregar(erro, msg):
    if erro:
        print("ERRO:", erro)
    else:
        print(f"ok  chave={msg.key().decode()}  partição={msg.partition()}  offset={msg.offset()}")

evento = {"pedido_id": "o-001", "cliente": "ana", "valor": 120.0}
produtor.produce(TOPICO, key="ana", value=json.dumps(evento), callback=ao_entregar)
produtor.flush()

## 2. Mesma chave → mesma partição
A partição é escolhida por `hash(chave) % número_de_partições`. Vamos mandar 60 pedidos de 5 clientes.

In [ ]:
clientes = ["ana", "bruno", "carla", "diego", "elis"]
destino = defaultdict(set)

def registrar(erro, msg):
    if not erro:
        destino[msg.key().decode()].add(msg.partition())

for i in range(60):
    c = clientes[i % len(clientes)]
    produtor.produce(TOPICO, key=c, value=json.dumps({"pedido_id": f"o-{i:03d}", "cliente": c}), callback=registrar)
produtor.flush()

for c in clientes:
    print(f"{c:6s} -> partições {sorted(destino[c])}")

Cada cliente caiu numa única partição. Como a ordem só é garantida **dentro** de uma partição,
todos os pedidos da Ana serão lidos na ordem em que foram escritos.

**E sem chave?** O produtor usa o *sticky partitioner*: enche um lote numa partição e depois troca.

In [ ]:
contagem = Counter()
for i in range(3000):
    produtor.produce(TOPICO, value=f"sem-chave-{i}",
                     callback=lambda e, m: contagem.update([m.partition()]) if not e else None)
produtor.flush()
print(sorted(contagem.items()))

## 3. Latência × throughput: `linger.ms` e `batch.size`
O produtor agrupa mensagens em lotes. Esperar um pouco (`linger.ms`) gera lotes maiores:
**mais throughput, mais latência**. Vamos medir.

O teste escreve no tópico **`aula3-benchmark`**, para não misturar 250 mil mensagens de teste com os pedidos que o Lab 3 vai ler.

In [ ]:
TOPICO_TESTE = "aula3-benchmark"   # tópico separado para não poluir aula3-pedidos

def medir(n=50_000, **cfg):
    p = Producer({"bootstrap.servers": BOOTSTRAP, **cfg})
    latencias = []
    payload = json.dumps({"pedido_id": "o-teste", "cliente": "benchmark", "itens": ["produto"] * 20, "valor": 199.9})
    inicio = time.perf_counter()
    for i in range(n):
        t0 = time.perf_counter()
        while True:
            try:
                p.produce(TOPICO_TESTE, value=payload,
                          callback=lambda e, m, t0=t0: latencias.append(time.perf_counter() - t0))
                break
            except BufferError:
                p.poll(0.05)
        p.poll(0)
    p.flush()
    total = time.perf_counter() - inicio
    latencias.sort()
    return {"msgs/s": round(n / total), "lat_p50_ms": round(latencias[len(latencias)//2] * 1000, 1),
            "lat_p99_ms": round(latencias[int(len(latencias)*.99)] * 1000, 1)}

for cfg in [{"linger.ms": 0, "acks": "1"},
            {"linger.ms": 20, "acks": "1"},
            {"linger.ms": 100, "acks": "1", "batch.size": 262144},
            {"linger.ms": 20, "acks": "all"},
            {"linger.ms": 20, "acks": "all", "compression.type": "lz4"}]:
    print(cfg, medir(**cfg))

Discuta com a turma: por que `acks=all` aumenta a latência? Quando vale a pena pagar esse preço?

## 4. Idempotência
Com `enable.idempotence=True` o broker descarta reenvios duplicados causados por *retries*
(cada produtor tem um id e números de sequência por partição). Isso dá **exactly-once na escrita
de um produtor para uma partição**. Exactly-once de ponta a ponta precisa de transações
(ou de um sink idempotente), que veremos com o Spark.

## Exercícios
1. Envie 10 mensagens com a chave `"zeca"` e confirme pela Kafka UI que todas estão na mesma partição.
2. Acrescente cabeçalhos (`headers=[("origem", b"lab2")]`) às mensagens e encontre-os na Kafka UI.
3. Rode `medir()` com `linger.ms` = 0, 5, 50 e 200 e monte um gráfico (matplotlib) de msgs/s × p99.
4. Pelo terminal, produza manualmente com chave:
   `kafka-console-producer.sh --bootstrap-server kafka1:9092 --topic aula3-pedidos --property parse.key=true --property key.separator=:`
   e digite linhas como `ana:{"valor": 10}`.